<a href="https://colab.research.google.com/github/deepanrajm/GL/blob/main/LLM/Agents/langgraph_basic_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langgraph langchain langchain_community langchain_experimental langchain_openai google-search-results numexpr

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32010 sha256=3fe7ed700c9938715d50537b79a3d48f3cdb255d9f4b2086ffb0edbbbe68a2ea
  Stored in directory: /root/.cache/pip/wheels/0c/47/f5/89b7e770ab2996baf8c910e7353d6391e373075a0ac213519e
Successfully built google-search-results
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4

In [3]:
import os
from typing import TypedDict, Literal

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SerpAPIWrapper
from langchain_classic.chains import LLMMathChain
from langgraph.graph import StateGraph, END

In [25]:
# --- Environment and LLM Setup ---
os.environ["SERPAPI_API_KEY"] = "db2270af85b9be502ee9357b9c6fcf639c8dbb50eba8da99e6cff24724d67e00"

llm = ChatOpenAI(openai_api_base = "https://openrouter.ai/api/v1", openai_api_key = "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", model = "google/gemma-3-4b-it:free",temperature=0.9) #max_tokens=100

# --- Tool Definitions ---
search_tool = SerpAPIWrapper()
math_tool = LLMMathChain.from_llm(llm=llm, verbose=False)

In [16]:
# --- State Definition ---
# Updated state to include a route for travel and a result.
class AgentState(TypedDict):
    topic: str
    explanation: str
    summary: str
    calculator_result: str
    travel_result: str
    route: Literal["research", "calculate", "travel"] # The planner's decision

In [17]:
def planner_agent(state: AgentState) -> dict:
    """
    This agent decides whether the user's topic is a math, research, or travel question.
    """
    print("---PLANNER---")
    topic = state["topic"]

    prompt = ChatPromptTemplate.from_template(
        """You are a planner agent. Your job is to determine the best path to take based on the user's query.

        Query: {topic}

        Based on the query, is this a math problem, a travel-related query (like finding hotels or flights), or a general research question?
        Respond with 'calculate', 'travel', or 'research'.
        """
    )

    chain = prompt | llm
    result = chain.invoke({"topic": topic})
    decision = result.content.strip().lower()

    print(f"Planner's Decision: {decision}")
    if "calculate" in decision:
        return {"route": "calculate"}
    elif "travel" in decision:
        return {"route": "travel"}
    else:
        return {"route": "research"}

In [18]:
def researcher_agent(state: AgentState) -> dict:
    """
    This agent uses a web search tool to find information on a topic and then explains it.
    """
    print("---RESEARCHER (with SerpApi Web Search)---")
    topic = state["topic"]

    search_results = search_tool.run(topic)
    print(f"Search Results:\n{search_results}")

    prompt = ChatPromptTemplate.from_template(
        """You are a helpful research assistant. Based on the following search results,
        provide a brief, easy-to-understand explanation of the topic: {topic}.

        Search Results:
        {search_results}
        """
    )

    chain = prompt | llm
    result = chain.invoke({"topic": topic, "search_results": search_results})

    print(f"Researcher's Explanation:\n{result.content}")
    return {"explanation": result.content}

In [19]:
def calculator_agent(state: AgentState) -> dict:
    """
    This agent uses an LLM Math tool to solve a math problem.
    """
    print("---CALCULATOR---")
    topic = state["topic"]

    result = math_tool.invoke({"question": topic})
    answer = result.get('answer', 'Could not calculate answer.')

    print(f"Calculator's Result: {answer}")
    return {"calculator_result": answer}

In [20]:
def travel_planner_agent(state: AgentState) -> dict:
    """
    This agent searches for travel-related information like hotels or flights.
    """
    print("---TRAVEL PLANNER---")
    topic = state["topic"]

    # For this example, we'll just use the general search tool.
    # A real-world application would use a dedicated travel API.
    search_results = search_tool.run(f"Find information about: {topic}")

    print(f"Travel Search Results:\n{search_results}")

    prompt = ChatPromptTemplate.from_template(
        """You are a helpful travel assistant. Summarize the following information for the user's travel query about '{topic}'.

        Search Results:
        {search_results}
        """
    )

    chain = prompt | llm
    result = chain.invoke({"topic": topic, "search_results": search_results})

    print(f"Travel Planner's Response:\n{result.content}")
    return {"travel_result": result.content}

In [21]:
def summarizer_agent(state: AgentState) -> dict:
    """
    This agent takes an explanation and summarizes it in one sentence.
    """
    print("---SUMMARIZER---")
    explanation = state["explanation"]

    prompt = ChatPromptTemplate.from_template(
        "You are a summarization expert. Condense the following text into a single, concise sentence:\n\n{explanation}"
    )

    chain = prompt | llm
    result = chain.invoke({"explanation": explanation})

    print(f"Summarizer's Output:\n{result.content}")
    return {"summary": result.content}

In [22]:
# --- Conditional Routing Function ---
def should_route(state: AgentState) -> Literal["research", "calculate", "travel"]:
    """This function is the decision point for our conditional edge."""
    return state["route"]

In [23]:
# --- Graph Definition ---

workflow = StateGraph(AgentState)

# Add all the agent functions as nodes
workflow.add_node("planner", planner_agent)
workflow.add_node("researcher", researcher_agent)
workflow.add_node("calculator", calculator_agent)
workflow.add_node("travel_planner", travel_planner_agent)
workflow.add_node("summarizer", summarizer_agent)

# Set the entry point to the planner
workflow.set_entry_point("planner")

# Add the conditional edge for the planner
workflow.add_conditional_edges(
    "planner",
    should_route,
    {
        "research": "researcher",
        "calculate": "calculator",
        "travel": "travel_planner",
    },
)

# Define the standard edges
workflow.add_edge("researcher", "summarizer")
workflow.add_edge("summarizer", END)
workflow.add_edge("calculator", END)
workflow.add_edge("travel_planner", END) # Travel result is final

# Compile the graph
app = workflow.compile()

In [26]:
# --- Main Execution Block ---
if __name__ == "__main__":
    topic = input("Please enter a topic, math problem, or travel query: ")

    inputs = {"topic": topic}
    final_state = app.invoke(inputs)

    print("\n--- FINAL RESULT ---")
    # The final result depends on which path was taken
    if final_state.get('summary'):
        print(final_state['summary'])
    elif final_state.get('calculator_result'):
        print(final_state['calculator_result'])
    elif final_state.get('travel_result'):
        print(final_state['travel_result'])
    else:
        print("An unexpected error occurred.")

Please enter a topic, math problem, or travel query: Give a travel itinerary for Japan for 15 days
---PLANNER---
Planner's Decision: travel
---TRAVEL PLANNER---
Travel Search Results:
['The following is a suggested itinerary for first time travelers to Japan who spend 15 days and 14 nights in Japan, and arrive/depart from Tokyo ...', 'This blog will provide you with valuable first-hand tips to make the perfect 15 days in Japan itinerary, with all the dos and donts and must-see', "I'm going to Japan for 14 nights/15 days from mid to late September. Traveling as a late 20's couple, first time in Japan: Sept 11 (day 1) - Tokyo (land 2:30 ...", 'Day 1: First taste of Tokyo · Day 2: Explore Tokyo · Day 3: Explore further from Tokyo · Day 4: Hakone day trip · Day 5: Takayama old town · Day 6: Shirakawago day ...', 'So far this is how our trip is structured: Day 1-4 in Tokyo Day 5 traveling tokyo-Osaka Day 6-8 in Osaka Day 9 traveling from Osaka to Kyoto Day ...', 'Days 1-4: Tokyo pop culture